In [1]:
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer


In [3]:
# Define the path to the model
model_path = r"C:\Users\Makai\Downloads\best_roberta_model.pt"

# Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Load the model architecture (binary classification)
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

# Load the trained weights
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()  # Set model to evaluation mode

# Function to predict
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()
    return "Class 1" if prediction == 1 else "Class 0"

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Example usage
text = "This is a sample text for classification."
print(f"Prediction: {predict(text)}")

Prediction: Class 1


In [8]:
# Example usage
text = "asdfKHFLSDKJHFKLSDJHkjlsdhkjfsLKJDFHKLSJFHLKSDJ"
print(f"Prediction: {predict(text)}")

Prediction: Class 1


In [23]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    print(f"Logits: {logits}")
    print(f"Probabilities: {probabilities}")
    # Default threshold of 0.5
    return "Class 1" if probabilities[0][1] > 0.8 else "Class 0"

In [ ]:
# Read input lines from in.txt
input_file = "2025-02-11-output.txt"
output_file = "trim.txt"

# Open the input file and read all lines
with open(input_file, "r", encoding="utf-8") as infile:
    lines = infile.readlines()

# Prepare a list to store lines classified as "Class 1"
class_1_lines = []

# Iterate through each line and make predictions
for line in lines:
    text = line.strip()  # Remove leading/trailing whitespace
    
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    
    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract logits and calculate probabilities
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)
    
    # Determine the predicted class
    prediction = "Class 1" if probabilities[0][1] > 0.55 else "Class 0"
    
    # If classified as "Class 1", add to the output list
    if prediction == "Class 1":
        class_1_lines.append(text)

# Write the "Class 1" lines to trim.txt
with open(output_file, "w", encoding="utf-8") as outfile:
    outfile.write("\n".join(class_1_lines))

print(f"Lines classified as 'Class 1' have been written to {output_file}.")

Lines classified as 'Class 1' have been written to trim.txt.
